# ABC2Vec — Notebook 03: Model Architecture

Defines the complete ABC2Vec Transformer encoder architecture:

1. **Patch Embedding** — bar-level character aggregation → d_model vectors
2. **Transformer Encoder** — 6-layer Transformer with self-attention
3. **Tune-level Pooling** — aggregate bar embeddings into a single tune embedding
4. **Projection Heads** — for different pre-training objectives
5. **Full ABC2Vec Model** — end-to-end architecture

Architecture based on CLaMP's M3 encoder, adapted for folk music:
```
ABC Notation → Bar Patchifier → Patch Embedding → Transformer Encoder → Pooling → Tune Embedding
```

In [1]:
import os, sys, json, math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import numpy as np
import pandas as pd
from rich import print as rprint

PROJECT_DIR = Path('/Volumes/LLModels/ABC2VEC')
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'

# Add project to path for imports
sys.path.insert(0, str(PROJECT_DIR))
from abc2vec.tokenizer import ABCVocabulary, BarPatchifier, PatchEmbedding

In [2]:
# Load config and vocabulary
vocab = ABCVocabulary.load(PROCESSED_DIR / 'vocab.json')
with open(PROCESSED_DIR / 'tokenizer_config.json') as f:
    tok_config = json.load(f)

print(f"Vocab size: {vocab.size}")
print(f"Tokenizer config: {tok_config}")

Vocab size: 98
Tokenizer config: {'vocab_size': 98, 'max_bar_length': 64, 'max_bars': 64, 'd_char': 64, 'd_model': 256, 'pad_idx': 0, 'mask_idx': 4, 'cls_idx': 2, 'sep_idx': 3}


## 3.1 Model Configuration

In [3]:
from dataclasses import dataclass, asdict

@dataclass
class ABC2VecConfig:
    """Configuration for ABC2Vec model."""
    # Vocabulary & patching
    vocab_size: int = 128
    max_bar_length: int = 64
    max_bars: int = 64
    pad_idx: int = 0
    mask_idx: int = 4
    
    # Patch embedding
    d_char: int = 64             # Character embedding dimension
    
    # Transformer encoder
    d_model: int = 256           # Hidden dimension
    n_heads: int = 8             # Attention heads
    n_layers: int = 6            # Transformer layers
    d_ff: int = 1024             # Feed-forward dimension
    dropout: float = 0.1
    
    # Embedding output
    d_embed: int = 128           # Final tune embedding dimension
    
    # Training
    mask_ratio: float = 0.15     # Ratio of bars to mask for MMM
    temperature: float = 0.07    # Contrastive loss temperature
    
    def save(self, path):
        with open(path, 'w') as f:
            json.dump(asdict(self), f, indent=2)
    
    @classmethod
    def load(cls, path):
        with open(path) as f:
            return cls(**json.load(f))

# Create config from tokenizer config
config = ABC2VecConfig(
    vocab_size=tok_config['vocab_size'],
    max_bar_length=tok_config['max_bar_length'],
    max_bars=tok_config['max_bars'],
    pad_idx=tok_config['pad_idx'],
    mask_idx=tok_config['mask_idx'],
    d_char=tok_config['d_char'],
    d_model=256,
    n_heads=8,
    n_layers=6,
    d_ff=1024,
    d_embed=128,
)

print("ABC2Vec Configuration:")
for k, v in asdict(config).items():
    print(f"  {k}: {v}")

# Save config
config.save(PROCESSED_DIR / 'model_config.json')
print(f"\nConfig saved to {PROCESSED_DIR / 'model_config.json'}")

ABC2Vec Configuration:
  vocab_size: 98
  max_bar_length: 64
  max_bars: 64
  pad_idx: 0
  mask_idx: 4
  d_char: 64
  d_model: 256
  n_heads: 8
  n_layers: 6
  d_ff: 1024
  dropout: 0.1
  d_embed: 128
  mask_ratio: 0.15
  temperature: 0.07

Config saved to /Volumes/LLModels/ABC2VEC/data/processed/model_config.json


## 3.2 Transformer Encoder

Standard Transformer encoder with pre-norm (more stable training).

In [4]:
class TransformerEncoderLayer(nn.Module):
    """Pre-norm Transformer encoder layer."""
    
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: (batch, seq_len) — True for real tokens, False for padding
        """
        # Convert mask for MultiheadAttention: True = ignore
        key_padding_mask = ~mask if mask is not None else None
        
        # Pre-norm self-attention
        residual = x
        x = self.norm1(x)
        attn_out, _ = self.self_attn(x, x, x, key_padding_mask=key_padding_mask)
        x = residual + self.dropout(attn_out)
        
        # Pre-norm feed-forward
        residual = x
        x = self.norm2(x)
        x = residual + self.feed_forward(x)
        
        return x


class ABC2VecEncoder(nn.Module):
    """
    Full ABC2Vec Transformer encoder.
    
    Takes bar-patched input and produces:
    - Bar-level embeddings (for masked modeling)
    - Tune-level embedding (for contrastive objectives)
    """
    
    def __init__(self, config: ABC2VecConfig):
        super().__init__()
        self.config = config
        
        # Patch embedding layer
        self.patch_embed = PatchEmbedding(
            vocab_size=config.vocab_size,
            d_char=config.d_char,
            d_model=config.d_model,
            max_bar_length=config.max_bar_length,
            max_bars=config.max_bars,
            pad_idx=config.pad_idx,
        )
        
        # Transformer layers
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(
                d_model=config.d_model,
                n_heads=config.n_heads,
                d_ff=config.d_ff,
                dropout=config.dropout
            )
            for _ in range(config.n_layers)
        ])
        
        # Final layer norm
        self.final_norm = nn.LayerNorm(config.d_model)
        
        # Tune-level projection head (bar embeddings → tune embedding)
        self.tune_projection = nn.Sequential(
            nn.Linear(config.d_model, config.d_model),
            nn.GELU(),
            nn.Linear(config.d_model, config.d_embed),
        )
    
    def encode_bars(self, bar_indices: torch.Tensor, char_mask: torch.Tensor,
                    bar_mask: torch.Tensor) -> torch.Tensor:
        """
        Encode bar-patched input to bar-level embeddings.
        
        Args:
            bar_indices: (batch, max_bars, max_bar_length)
            char_mask: (batch, max_bars, max_bar_length)
            bar_mask: (batch, max_bars)
        
        Returns:
            bar_embeddings: (batch, max_bars, d_model)
        """
        # Patch embedding
        x, mask = self.patch_embed(bar_indices, char_mask, bar_mask)
        
        # Transformer layers
        for layer in self.layers:
            x = layer(x, mask)
        
        x = self.final_norm(x)
        return x
    
    def get_tune_embedding(self, bar_indices: torch.Tensor, char_mask: torch.Tensor,
                           bar_mask: torch.Tensor) -> torch.Tensor:
        """
        Get tune-level embedding via mean pooling + projection.
        
        Returns:
            tune_embedding: (batch, d_embed)
        """
        bar_embeddings = self.encode_bars(bar_indices, char_mask, bar_mask)
        
        # Mean pool over non-padding bars
        mask_expanded = bar_mask.unsqueeze(-1).float()  # (batch, max_bars, 1)
        pooled = (bar_embeddings * mask_expanded).sum(dim=1)  # (batch, d_model)
        bar_count = bar_mask.sum(dim=1, keepdim=True).float().clamp(min=1)  # (batch, 1)
        pooled = pooled / bar_count
        
        # Project to embedding dimension
        tune_emb = self.tune_projection(pooled)  # (batch, d_embed)
        
        # L2 normalize for cosine similarity
        tune_emb = F.normalize(tune_emb, p=2, dim=-1)
        
        return tune_emb
    
    def forward(self, bar_indices: torch.Tensor, char_mask: torch.Tensor,
                bar_mask: torch.Tensor) -> dict:
        """
        Full forward pass returning both bar-level and tune-level representations.
        """
        bar_embeddings = self.encode_bars(bar_indices, char_mask, bar_mask)
        
        # Mean pool
        mask_expanded = bar_mask.unsqueeze(-1).float()
        pooled = (bar_embeddings * mask_expanded).sum(dim=1)
        bar_count = bar_mask.sum(dim=1, keepdim=True).float().clamp(min=1)
        pooled = pooled / bar_count
        
        # Tune embedding
        tune_emb = F.normalize(self.tune_projection(pooled), p=2, dim=-1)
        
        return {
            'bar_embeddings': bar_embeddings,   # (batch, max_bars, d_model)
            'pooled': pooled,                    # (batch, d_model)
            'tune_embedding': tune_emb,          # (batch, d_embed)
        }

# Create encoder
encoder = ABC2VecEncoder(config)
print(f"ABC2Vec Encoder created")
print(f"  Total parameters: {sum(p.numel() for p in encoder.parameters()):,}")
print(f"  Trainable parameters: {sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}")

ABC2Vec Encoder created
  Total parameters: 4,877,824
  Trainable parameters: 4,877,824


## 3.3 Masked Music Modeling (MMM) Head

BERT-style masked prediction at the bar level.

In [5]:
class MaskedMusicModelingHead(nn.Module):
    """
    Predicts masked bar patches from their Transformer representations.
    
    For each masked bar, predicts the characters that were in that bar.
    Uses the bar-level embedding to reconstruct the character sequence.
    """
    
    def __init__(self, config: ABC2VecConfig):
        super().__init__()
        self.config = config
        
        # Transform bar embedding to character predictions
        self.bar_to_chars = nn.Sequential(
            nn.Linear(config.d_model, config.d_model),
            nn.GELU(),
            nn.LayerNorm(config.d_model),
        )
        
        # Predict character at each position in the bar
        self.char_predictor = nn.Linear(config.d_model, config.vocab_size)
        
        # Learnable position embeddings for characters within a bar
        self.char_pos = nn.Embedding(config.max_bar_length, config.d_model)
    
    def forward(self, bar_embeddings: torch.Tensor, masked_indices: torch.Tensor) -> torch.Tensor:
        """
        Args:
            bar_embeddings: (batch, max_bars, d_model) — encoder output
            masked_indices: (batch, num_masked) — indices of masked bars
        
        Returns:
            char_logits: (batch, num_masked, max_bar_length, vocab_size)
        """
        batch_size = bar_embeddings.shape[0]
        num_masked = masked_indices.shape[1]
        
        # Gather masked bar embeddings
        # masked_indices: (batch, num_masked) → expand for gathering
        idx = masked_indices.unsqueeze(-1).expand(-1, -1, bar_embeddings.shape[-1])
        masked_embeds = torch.gather(bar_embeddings, dim=1, index=idx)  # (batch, num_masked, d_model)
        
        # Transform
        hidden = self.bar_to_chars(masked_embeds)  # (batch, num_masked, d_model)
        
        # Expand to character positions
        # hidden: (batch, num_masked, 1, d_model) + char_pos: (1, 1, max_bar_length, d_model)
        char_positions = self.char_pos.weight.unsqueeze(0).unsqueeze(0)  # (1, 1, max_bar_length, d_model)
        hidden_expanded = hidden.unsqueeze(2) + char_positions  # (batch, num_masked, max_bar_length, d_model)
        
        # Predict characters
        char_logits = self.char_predictor(hidden_expanded)  # (batch, num_masked, max_bar_length, vocab_size)
        
        return char_logits

mmm_head = MaskedMusicModelingHead(config)
print(f"MMM Head parameters: {sum(p.numel() for p in mmm_head.parameters()):,}")

MMM Head parameters: 107,874


## 3.4 Complete ABC2Vec Model

Combines encoder + all heads into a single model.

In [6]:
class ABC2VecModel(nn.Module):
    """
    Complete ABC2Vec model with:
    - Transformer encoder (bar-level embeddings)
    - Masked Music Modeling head
    - Tune-level embedding (for contrastive objectives)
    """
    
    def __init__(self, config: ABC2VecConfig):
        super().__init__()
        self.config = config
        
        # Core encoder
        self.encoder = ABC2VecEncoder(config)
        
        # MMM head
        self.mmm_head = MaskedMusicModelingHead(config)
    
    def mask_bars(self, bar_indices: torch.Tensor, bar_mask: torch.Tensor,
                  mask_ratio: float = None) -> tuple:
        """
        Apply random bar masking for MMM objective.
        
        Returns:
            masked_bar_indices: bar_indices with some bars zeroed out
            masked_positions: (batch, num_masked) indices of masked bars
            target_bar_indices: original bar_indices for the masked positions
        """
        if mask_ratio is None:
            mask_ratio = self.config.mask_ratio
        
        batch_size, max_bars, max_bar_length = bar_indices.shape
        device = bar_indices.device
        
        masked_bar_indices = bar_indices.clone()
        all_masked_positions = []
        all_targets = []
        
        # Determine num bars to mask per sample
        num_real_bars = bar_mask.sum(dim=1).long()  # (batch,)
        num_to_mask = (num_real_bars.float() * mask_ratio).long().clamp(min=1)
        max_masked = num_to_mask.max().item()
        
        masked_positions = torch.zeros(batch_size, max_masked, dtype=torch.long, device=device)
        target_bar_indices = torch.zeros(
            batch_size, max_masked, max_bar_length, dtype=torch.long, device=device
        )
        mask_labels = torch.zeros(batch_size, max_masked, dtype=torch.bool, device=device)
        
        for b in range(batch_size):
            n_real = num_real_bars[b].item()
            n_mask = num_to_mask[b].item()
            
            # Random selection of bars to mask
            perm = torch.randperm(n_real, device=device)[:n_mask]
            
            masked_positions[b, :n_mask] = perm
            mask_labels[b, :n_mask] = True
            
            for i, pos in enumerate(perm):
                target_bar_indices[b, i] = bar_indices[b, pos]
                # Replace masked bar with zeros (pad tokens)
                masked_bar_indices[b, pos] = self.config.pad_idx
        
        return masked_bar_indices, masked_positions, target_bar_indices, mask_labels
    
    def forward_mmm(self, bar_indices, char_mask, bar_mask):
        """
        Forward pass for Masked Music Modeling.
        
        Returns dict with 'mmm_logits', 'mmm_targets', 'mmm_mask'.
        """
        # Apply masking
        masked_indices, masked_pos, targets, mask_labels = self.mask_bars(
            bar_indices, bar_mask
        )
        
        # Encode with masked bars
        masked_char_mask = char_mask.clone()
        for b in range(bar_indices.shape[0]):
            for pos in masked_pos[b][mask_labels[b]]:
                masked_char_mask[b, pos] = False
        
        bar_embeddings = self.encoder.encode_bars(
            masked_indices, masked_char_mask, bar_mask
        )
        
        # Predict masked bars
        char_logits = self.mmm_head(bar_embeddings, masked_pos)
        
        return {
            'mmm_logits': char_logits,       # (batch, num_masked, max_bar_length, vocab_size)
            'mmm_targets': targets,           # (batch, num_masked, max_bar_length)
            'mmm_mask': mask_labels,          # (batch, num_masked) — True for valid masked positions
            'bar_embeddings': bar_embeddings,
        }
    
    def forward_contrastive(self, bar_indices_1, char_mask_1, bar_mask_1,
                            bar_indices_2, char_mask_2, bar_mask_2):
        """
        Forward pass for contrastive objectives (transposition invariance, section contrast).
        
        Returns:
            emb_1: (batch, d_embed) — tune embeddings for first input
            emb_2: (batch, d_embed) — tune embeddings for second input
        """
        emb_1 = self.encoder.get_tune_embedding(bar_indices_1, char_mask_1, bar_mask_1)
        emb_2 = self.encoder.get_tune_embedding(bar_indices_2, char_mask_2, bar_mask_2)
        return emb_1, emb_2
    
    def get_embedding(self, bar_indices, char_mask, bar_mask):
        """
        Inference-time: get tune-level embedding.
        """
        return self.encoder.get_tune_embedding(bar_indices, char_mask, bar_mask)

# Create complete model
model = ABC2VecModel(config)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"ABC2Vec Model Summary:")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size:           ~{total_params * 4 / 1e6:.1f} MB (float32)")
print(f"")

# Parameter breakdown
print("Parameter breakdown:")
for name, module in model.named_children():
    n_params = sum(p.numel() for p in module.parameters())
    print(f"  {name}: {n_params:,} params")

ABC2Vec Model Summary:
  Total parameters:     4,985,698
  Trainable parameters: 4,985,698
  Model size:           ~19.9 MB (float32)

Parameter breakdown:
  encoder: 4,877,824 params
  mmm_head: 107,874 params


## 3.5 Test Forward Passes

In [7]:
# Create a dummy batch for testing
batch_size = 4
dummy_bar_indices = torch.randint(0, config.vocab_size, (batch_size, config.max_bars, config.max_bar_length))
dummy_char_mask = torch.ones(batch_size, config.max_bars, config.max_bar_length, dtype=torch.bool)
dummy_bar_mask = torch.ones(batch_size, config.max_bars, dtype=torch.bool)
# Simulate varying bar counts
dummy_bar_mask[0, 20:] = False
dummy_bar_mask[1, 16:] = False
dummy_bar_mask[2, 24:] = False
dummy_bar_mask[3, 32:] = False

model.eval()

# Test 1: Full forward (encoder + tune embedding)
with torch.no_grad():
    output = model.encoder(dummy_bar_indices, dummy_char_mask, dummy_bar_mask)
    
print("Test 1 — Encoder forward:")
for k, v in output.items():
    print(f"  {k}: {v.shape}")

# Test 2: MMM forward
with torch.no_grad():
    mmm_out = model.forward_mmm(dummy_bar_indices, dummy_char_mask, dummy_bar_mask)

print(f"\nTest 2 — MMM forward:")
for k, v in mmm_out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

# Test 3: Contrastive forward
with torch.no_grad():
    emb1, emb2 = model.forward_contrastive(
        dummy_bar_indices, dummy_char_mask, dummy_bar_mask,
        dummy_bar_indices, dummy_char_mask, dummy_bar_mask
    )

print(f"\nTest 3 — Contrastive forward:")
print(f"  emb1: {emb1.shape}, norm: {emb1.norm(dim=-1)}")
print(f"  emb2: {emb2.shape}, norm: {emb2.norm(dim=-1)}")
print(f"  Cosine similarity (same input): {F.cosine_similarity(emb1, emb2)}")

# Test 4: Get embedding (inference)
with torch.no_grad():
    tune_emb = model.get_embedding(dummy_bar_indices, dummy_char_mask, dummy_bar_mask)

print(f"\nTest 4 — Inference embedding:")
print(f"  Shape: {tune_emb.shape}")
print(f"  L2 norm: {tune_emb.norm(dim=-1)}")

Test 1 — Encoder forward:
  bar_embeddings: torch.Size([4, 64, 256])
  pooled: torch.Size([4, 256])
  tune_embedding: torch.Size([4, 128])

Test 2 — MMM forward:
  mmm_logits: torch.Size([4, 4, 64, 98])
  mmm_targets: torch.Size([4, 4, 64])
  mmm_mask: torch.Size([4, 4])
  bar_embeddings: torch.Size([4, 64, 256])

Test 3 — Contrastive forward:
  emb1: torch.Size([4, 128]), norm: tensor([1., 1., 1., 1.])
  emb2: torch.Size([4, 128]), norm: tensor([1., 1., 1., 1.])
  Cosine similarity (same input): tensor([1.0000, 1.0000, 1.0000, 1.0000])

Test 4 — Inference embedding:
  Shape: torch.Size([4, 128])
  L2 norm: tensor([1., 1., 1., 1.])


## 3.6 Export Model Module

In [8]:
model_code = '''
"""ABC2Vec Model Architecture."""

import json, math
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F

from .tokenizer import PatchEmbedding


@dataclass
class ABC2VecConfig:
    vocab_size: int = 128
    max_bar_length: int = 64
    max_bars: int = 64
    pad_idx: int = 0
    mask_idx: int = 4
    d_char: int = 64
    d_model: int = 256
    n_heads: int = 8
    n_layers: int = 6
    d_ff: int = 1024
    dropout: float = 0.1
    d_embed: int = 128
    mask_ratio: float = 0.15
    temperature: float = 0.07

    def save(self, path):
        with open(path, "w") as f:
            json.dump(asdict(self), f, indent=2)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            return cls(**json.load(f))


class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        key_padding_mask = ~mask if mask is not None else None
        residual = x
        x = self.norm1(x)
        attn_out, _ = self.self_attn(x, x, x, key_padding_mask=key_padding_mask)
        x = residual + self.dropout(attn_out)
        residual = x
        x = self.norm2(x)
        x = residual + self.feed_forward(x)
        return x


class ABC2VecEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.patch_embed = PatchEmbedding(
            config.vocab_size, config.d_char, config.d_model,
            config.max_bar_length, config.max_bars, config.pad_idx)
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(config.d_model, config.n_heads, config.d_ff, config.dropout)
            for _ in range(config.n_layers)])
        self.final_norm = nn.LayerNorm(config.d_model)
        self.tune_projection = nn.Sequential(
            nn.Linear(config.d_model, config.d_model), nn.GELU(),
            nn.Linear(config.d_model, config.d_embed))

    def encode_bars(self, bar_indices, char_mask, bar_mask):
        x, mask = self.patch_embed(bar_indices, char_mask, bar_mask)
        for layer in self.layers:
            x = layer(x, mask)
        return self.final_norm(x)

    def get_tune_embedding(self, bar_indices, char_mask, bar_mask):
        bar_embeddings = self.encode_bars(bar_indices, char_mask, bar_mask)
        mask_exp = bar_mask.unsqueeze(-1).float()
        pooled = (bar_embeddings * mask_exp).sum(dim=1) / bar_mask.sum(dim=1, keepdim=True).float().clamp(min=1)
        return F.normalize(self.tune_projection(pooled), p=2, dim=-1)

    def forward(self, bar_indices, char_mask, bar_mask):
        bar_embeddings = self.encode_bars(bar_indices, char_mask, bar_mask)
        mask_exp = bar_mask.unsqueeze(-1).float()
        pooled = (bar_embeddings * mask_exp).sum(dim=1) / bar_mask.sum(dim=1, keepdim=True).float().clamp(min=1)
        tune_emb = F.normalize(self.tune_projection(pooled), p=2, dim=-1)
        return {"bar_embeddings": bar_embeddings, "pooled": pooled, "tune_embedding": tune_emb}


class MaskedMusicModelingHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.bar_to_chars = nn.Sequential(
            nn.Linear(config.d_model, config.d_model), nn.GELU(),
            nn.LayerNorm(config.d_model))
        self.char_predictor = nn.Linear(config.d_model, config.vocab_size)
        self.char_pos = nn.Embedding(config.max_bar_length, config.d_model)

    def forward(self, bar_embeddings, masked_indices):
        idx = masked_indices.unsqueeze(-1).expand(-1, -1, bar_embeddings.shape[-1])
        masked_embeds = torch.gather(bar_embeddings, dim=1, index=idx)
        hidden = self.bar_to_chars(masked_embeds)
        char_positions = self.char_pos.weight.unsqueeze(0).unsqueeze(0)
        hidden_expanded = hidden.unsqueeze(2) + char_positions
        return self.char_predictor(hidden_expanded)


class ABC2VecModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.encoder = ABC2VecEncoder(config)
        self.mmm_head = MaskedMusicModelingHead(config)

    def mask_bars(self, bar_indices, bar_mask, mask_ratio=None):
        if mask_ratio is None:
            mask_ratio = self.config.mask_ratio
        batch_size, max_bars, max_bar_length = bar_indices.shape
        device = bar_indices.device
        masked_bar_indices = bar_indices.clone()
        num_real = bar_mask.sum(dim=1).long()
        num_to_mask = (num_real.float() * mask_ratio).long().clamp(min=1)
        max_masked = num_to_mask.max().item()
        masked_pos = torch.zeros(batch_size, max_masked, dtype=torch.long, device=device)
        targets = torch.zeros(batch_size, max_masked, max_bar_length, dtype=torch.long, device=device)
        mask_labels = torch.zeros(batch_size, max_masked, dtype=torch.bool, device=device)
        for b in range(batch_size):
            n = num_to_mask[b].item()
            perm = torch.randperm(num_real[b].item(), device=device)[:n]
            masked_pos[b, :n] = perm
            mask_labels[b, :n] = True
            for i, p in enumerate(perm):
                targets[b, i] = bar_indices[b, p]
                masked_bar_indices[b, p] = self.config.pad_idx
        return masked_bar_indices, masked_pos, targets, mask_labels

    def forward_mmm(self, bar_indices, char_mask, bar_mask):
        masked_indices, masked_pos, targets, mask_labels = self.mask_bars(bar_indices, bar_mask)
        masked_cm = char_mask.clone()
        for b in range(bar_indices.shape[0]):
            for pos in masked_pos[b][mask_labels[b]]:
                masked_cm[b, pos] = False
        bar_embeddings = self.encoder.encode_bars(masked_indices, masked_cm, bar_mask)
        logits = self.mmm_head(bar_embeddings, masked_pos)
        return {"mmm_logits": logits, "mmm_targets": targets, "mmm_mask": mask_labels,
                "bar_embeddings": bar_embeddings}

    def forward_contrastive(self, bi1, cm1, bm1, bi2, cm2, bm2):
        return (self.encoder.get_tune_embedding(bi1, cm1, bm1),
                self.encoder.get_tune_embedding(bi2, cm2, bm2))

    def get_embedding(self, bar_indices, char_mask, bar_mask):
        return self.encoder.get_tune_embedding(bar_indices, char_mask, bar_mask)
'''

model_module_path = PROJECT_DIR / 'abc2vec' / 'model.py'
model_module_path.write_text(model_code)
print(f"Model module saved to {model_module_path}")

Model module saved to /Volumes/LLModels/ABC2VEC/abc2vec/model.py
